# Physical LEGO EV3 Experimental Results
Analyse original controller logs. Cartesian positions are reconstructed from encoder angles using the kinematic model; they are not independent external position measurements. All historic plots and metrics remain distinct from any rerun. Logs are read from EV3_LOG_DIR and are not bundled. Recorded joint arrays are radians and are converted once to degrees for display.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import re
DATA_DIR = Path(os.environ.get('EV3_LOG_DIR', 'data'))
print('Log directory:', DATA_DIR)


In [ ]:
DATASET_NOTES = {'CENCI3': 'k_z = k*2.5', 'CENCI3z': 'k_z = k*2', 'CENCI3v': 'k_z = k*1.5'}
FILE_PATTERNS = {'dt': ['{prefix}_dt_reale_online.txt', '{prefix}_dt_reale.txt', '{prefix}_dt.txt'], 'Pd': ['{prefix}_Pd_xyz_online.txt', '{prefix}_Pd_xyz.txt', '{prefix}_Pd.txt'], 'Preal': ['{prefix}_Preal_xyz_online.txt', '{prefix}_Preal_xyz.txt', '{prefix}_Preal.txt'], 'Qdes': ['{prefix}_Qdes_online.txt', '{prefix}_Qdes.txt'], 'Qreal': ['{prefix}_Qreal_online.txt', '{prefix}_Qreal.txt'], 'Qdot': ['{prefix}_Qdot_online.txt', '{prefix}_Qdot.txt']}

def find_file(prefix, key):
    for pattern in FILE_PATTERNS[key]:
        candidate = DATA_DIR / pattern.format(prefix=prefix)
        if candidate.exists():
            return candidate
    return None

def load_1d(path):
    data = np.loadtxt(path)
    return np.atleast_1d(data).astype(float)

def load_2d(path, expected_cols=None):
    data = np.loadtxt(path)
    if data.ndim == 1:
        data = data.reshape(1, -1)
    if expected_cols is not None and data.shape[1] != expected_cols:
        raise ValueError(f'{path.name}: expected {expected_cols} columns, found {data.shape[1]}')
    return data.astype(float)

def discover_prefixes():
    prefixes = set()
    known_suffixes = ['_dt_reale_online.txt', '_dt_reale.txt', '_dt.txt', '_Pd_xyz_online.txt', '_Pd_xyz.txt', '_Pd.txt', '_Preal_xyz_online.txt', '_Preal_xyz.txt', '_Preal.txt', '_Qdes_online.txt', '_Qdes.txt', '_Qreal_online.txt', '_Qreal.txt', '_Qdot_online.txt', '_Qdot.txt']
    for path in DATA_DIR.glob('CENCI*.txt'):
        name = path.name
        for suffix in known_suffixes:
            if name.endswith(suffix):
                prefix = name[:-len(suffix)]
                prefixes.add(prefix)
                break
    return sorted(prefixes)

def build_time_vector(dt_file, N):
    if dt_file is None:
        raise ValueError('Timing log required; sample indices cannot be labelled as seconds')
    dt_real = load_1d(dt_file)
    dt_real = dt_real[:N]
    time = np.zeros(N)
    if N > 1:
        time[1:] = np.cumsum(dt_real[1:N])
    return time

def load_dataset(prefix):
    files = {key: find_file(prefix, key) for key in FILE_PATTERNS.keys()}
    required = ['Pd', 'Preal', 'Qdes', 'Qreal']
    missing = [key for key in required if files[key] is None]
    if missing:
        print(f'Dataset {prefix} skipped: missing {missing}')
        return None
    Pd = load_2d(files['Pd'], expected_cols=3)
    Preal = load_2d(files['Preal'], expected_cols=3)
    Qdes_deg = np.degrees(load_2d(files['Qdes'], expected_cols=4))
    Qreal_deg = np.degrees(load_2d(files['Qreal'], expected_cols=4))
    arrays_for_length = [Pd, Preal, Qdes_deg, Qreal_deg]
    Qdot = None
    if files['Qdot'] is not None:
        Qdot = load_2d(files['Qdot'], expected_cols=4)
        arrays_for_length.append(Qdot)
    if files['dt'] is not None:
        dt_real = load_1d(files['dt'])
        arrays_for_length.append(dt_real)
    else:
        dt_real = None
    N = min((len(a) for a in arrays_for_length))
    Pd = Pd[:N, :]
    Preal = Preal[:N, :]
    Qdes_deg = Qdes_deg[:N, :]
    Qreal_deg = Qreal_deg[:N, :]
    if Qdot is not None:
        Qdot = Qdot[:N, :]
    time = build_time_vector(files['dt'], N)
    err_xyz = Pd - Preal
    err_norm = np.linalg.norm(err_xyz, axis=1)
    return {'prefix': prefix, 'note': DATASET_NOTES.get(prefix, ''), 'files': files, 'time': time, 'Pd': Pd, 'Preal': Preal, 'Qdes_deg': Qdes_deg, 'Qreal_deg': Qreal_deg, 'Qdot': Qdot, 'err_xyz': err_xyz, 'err_norm': err_norm}
prefixes = discover_prefixes()
print('datasets found:', prefixes)
datasets = {}
for prefix in prefixes:
    dataset = load_dataset(prefix)
    if dataset is not None:
        datasets[prefix] = dataset
print('datasets loaded:', list(datasets.keys()))
for prefix, d in datasets.items():
    print(f"{prefix:10s} | N = {len(d['time']):4d} | duration = {d['time'][-1]:.3f} s | note: {d['note']}")
if not datasets:
    raise FileNotFoundError('No complete experimental logs found. Set EV3_LOG_DIR to the original log directory.')


## Desired versus encoder-derived position
Plot Cartesian components for the original gain trials.


In [ ]:
selected_prefixes = ['CENCI10', 'CENCI1_5', 'CENCI2']
labels = ['x', 'y', 'z']
for prefix in selected_prefixes:
    if prefix not in datasets:
        print(f'Dataset {prefix} not found, plot skipped.')
        continue
    d = datasets[prefix]
    time = d['time']
    Pd = d['Pd']
    Preal = d['Preal']
    note = f" ({d['note']})" if d['note'] else ''
    plt.figure(figsize=(12, 8))
    for i, ax_name in enumerate(labels):
        plt.subplot(3, 1, i + 1)
        plt.plot(time, Pd[:, i], 'r--', linewidth=2, label=f'{ax_name} desired')
        plt.plot(time, Preal[:, i], linewidth=2, label=f'{ax_name} encoder-derived')
        plt.ylabel(f'{ax_name} [m]')
        plt.grid(True)
        plt.legend()
    plt.xlabel('time [s]')
    plt.suptitle(f'{prefix}{note} - desired versus encoder-derived Cartesian position')
    plt.tight_layout()
    plt.show()


### Vertical tracking
The supplied presentation reports greater difficulty on Z and attributes it to gravity, friction, backlash and motor response. These are plausible explanations rather than independently measured causal effects.


In [ ]:
excluded_prefixes = ['CENCI10', 'CENCI1_5', 'CENCI2']
labels = ['x', 'y', 'z']
other_prefixes = [prefix for prefix in datasets.keys() if prefix not in excluded_prefixes]
print('Dataset plotted in this section:', other_prefixes)
for prefix in other_prefixes:
    d = datasets[prefix]
    time = d['time']
    Pd = d['Pd']
    Preal = d['Preal']
    note = f" ({d['note']})" if d['note'] else ''
    plt.figure(figsize=(12, 8))
    for i, ax_name in enumerate(labels):
        plt.subplot(3, 1, i + 1)
        plt.plot(time, Pd[:, i], 'r--', linewidth=2, label=f'{ax_name} desired')
        plt.plot(time, Preal[:, i], linewidth=2, label=f'{ax_name} encoder-derived')
        plt.ylabel(f'{ax_name} [m]')
        plt.grid(True)
        plt.legend()
    plt.xlabel('time [s]')
    plt.suptitle(f'{prefix}{note} - desired versus encoder-derived Cartesian position')
    plt.tight_layout()
    plt.show()


### Vertical gain configurations
Notebook annotations identify CENCI3 as kz=2.5k, CENCI3z as kz=2k and CENCI3v as kz=1.5k. Only the supplied controller verifies CENCI3v (k=3). Earlier controller variants were not supplied, so other mappings remain historical annotations.


## Tracking-error comparison
Compare Euclidean Cartesian error over the four named trials.


In [ ]:
selected_prefixes = ['CENCI10', 'CENCI3', 'CENCI3z', 'CENCI3v']
plt.figure(figsize=(12, 5))
for prefix in selected_prefixes:
    if prefix not in datasets:
        print(f'Dataset {prefix} not found, plot skipped.')
        continue
    d = datasets[prefix]
    label = prefix
    if d['note']:
        label += f" - {d['note']}"
    plt.plot(d['time'], d['err_norm'] * 100, linewidth=2, label=label)
plt.xlabel('time [s]')
plt.ylabel('||error|| [cm]')
plt.title('Cartesian error-norm comparison: CENCI10, CENCI3, CENCI3z, CENCI3v')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
selected_prefixes = ['CENCI10', 'CENCI3', 'CENCI3z', 'CENCI3v']
error_summary = []
for prefix in selected_prefixes:
    if prefix not in datasets:
        print(f'Dataset {prefix} not found, skipped.')
        continue
    d = datasets[prefix]
    err_mean_cm = np.mean(d['err_norm']) * 100
    err_max_cm = np.max(d['err_norm']) * 100
    err_final_cm = d['err_norm'][-1] * 100
    error_summary.append((prefix, err_mean_cm, err_max_cm, err_final_cm))
error_summary = sorted(error_summary, key=lambda x: x[1])
print('=== MEAN CARTESIAN ERROR FOR SELECTED DATASETS ===')
print()
for prefix, err_mean_cm, err_max_cm, err_final_cm in error_summary:
    note = datasets[prefix]['note']
    note_txt = f' | {note}' if note else ''
    print(f'{prefix}{note_txt}')
    print(f'  mean error   = {err_mean_cm:.3f} cm')
    print(f'  maximum error = {err_max_cm:.3f} cm')
    print(f'  final error  = {err_final_cm:.3f} cm')
    print()
if len(error_summary) > 0:
    best_prefix = error_summary[0][0]
    best_err = error_summary[0][1]
    print('Dataset with lowest mean error among selected datasets:')
    print(f'{best_prefix} with mean error = {best_err:.3f} cm')
else:
    print('no selected dataset found.')


## Commanded and encoder joint positions
Display radians from the logs as degrees after explicit conversion.


In [ ]:
selected_prefixes = ['CENCI10', 'CENCI3', 'CENCI3z', 'CENCI3v']
for prefix in selected_prefixes:
    if prefix not in datasets:
        print(f'Dataset {prefix} not found, plot skipped.')
        continue
    d = datasets[prefix]
    time = d['time']
    Qdes_deg = d['Qdes_deg']
    Qreal_deg = d['Qreal_deg']
    note = f" ({d['note']})" if d['note'] else ''
    plt.figure(figsize=(12, 9))
    for j in range(4):
        plt.subplot(4, 1, j + 1)
        plt.plot(time, Qdes_deg[:, j], 'r--', linewidth=2, label=f'q{j + 1} commanded')
        plt.plot(time, Qreal_deg[:, j], linewidth=2, label=f'q{j + 1} encoder-derived')
        plt.ylabel(f'q{j + 1} [deg]')
        plt.grid(True)
        plt.legend()
    plt.xlabel('time [s]')
    plt.suptitle(f'{prefix}{note} - commanded versus encoder joint positions')
    plt.tight_layout()
    plt.show()


In [ ]:
dataset_a = 'CENCI3v'
dataset_b = 'CENCI10'
if dataset_a not in datasets:
    raise ValueError(f'Dataset {dataset_a} not found. check the input log files.')
if dataset_b not in datasets:
    raise ValueError(f'Dataset {dataset_b} not found. check the input log files.')
d_a = datasets[dataset_a]
d_b = datasets[dataset_b]
for j in range(4):
    plt.figure(figsize=(12, 5))
    plt.plot(d_a['time'], d_a['Qreal_deg'][:, j], linewidth=2, label=f'{dataset_a} - q{j + 1} encoder-derived')
    plt.plot(d_b['time'], d_b['Qreal_deg'][:, j], linewidth=2, label=f'{dataset_b} - q{j + 1} encoder-derived')
    plt.plot(d_a['time'], d_a['Qdes_deg'][:, j], '--', linewidth=1.5, label=f'{dataset_a} - q{j + 1} command')
    plt.plot(d_b['time'], d_b['Qdes_deg'][:, j], '--', linewidth=1.5, label=f'{dataset_b} - q{j + 1} command')
    plt.xlabel('time [s]')
    plt.ylabel(f'q{j + 1} [deg]')
    plt.title(f'comparison joint {j + 1}: {dataset_a} vs {dataset_b}')
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()


## Gain tradeoff
The original presentation interprets K=10 as lower Cartesian error with more abrupt joint motion, especially q4, while CENCI3v is smoother. This is a qualitative hardware observation, not a quantified stability proof. Historical degree-labelled plots were affected by a missing radian-to-degree conversion; new plots correct their scale.


## Verified original physical-trial results

Original log-derived mean/max/final Cartesian errors (cm): CENCI10 0.482/1.736/0.337; CENCI3v 0.597/1.885/0.177; CENCI3z 0.862/2.526/0.665; CENCI3 0.950/3.102/0.828. These values were checked from the original local logs, not new physical trials. Cartesian positions are encoder-derived.

![Verified project result](../assets/experiments/cell_9_figure_0.png)

![Verified project result](../assets/experiments/cell_6_figure_1.png)